# ImageCLEF 2026 – Deepfake Detection v4 (Full)
### 실행 전: 런타임 > 런타임 유형 변경 > **T4 GPU** 선택

| 섹션 | 신호 | 방식 |
|------|------|----- |
| 이미지 | CLIP KNN | 실제 데이터 분포 거리 |
| 이미지 | CLIP 제로샷 | 텍스트 프롬프트 유사도 |
| 이미지 | dima806 + TTA | 딥페이크 전용 모델 + 좌우반전 평균 |
| 이미지 | FFT 주파수 | GAN/업샘플링 아티팩트 탐지 |
| 오디오 | WavLM-Large KNN | 실제 오디오 분포 거리 |
| 오디오 | wav2vec2 KNN | 두 번째 음성 모델 앙상블 |

**예상 시간: 3~4시간**

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers torchaudio scikit-learn pillow pandas tqdm accelerate opencv-python-headless
import torch, os, glob, shutil, zipfile
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

## 1. 데이터 압축 해제 + 실제 데이터 전처리

In [ ]:
TEST_ZIP = '/content/drive/MyDrive/ImageCLEF2026-DeepFakeDetection-Tes.zip'
REAL_ZIP = '/content/drive/MyDrive/Real_Data_Generation_Task.zip'
TEST_DIR = '/content/test_data'
REAL_DIR = '/content/real_data'

def extract(zip_path, out_dir):
    if not os.path.exists(out_dir):
        print(f'압축 해제: {os.path.basename(zip_path)} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(out_dir)
        print('완료!')
    else:
        print(f'이미 해제됨: {out_dir}')

extract(TEST_ZIP, TEST_DIR)
extract(REAL_ZIP, REAL_DIR)

In [ ]:
# 오디오: reference_set.zip 추가 해제
inner_zip = glob.glob(os.path.join(REAL_DIR, '**', 'reference_set.zip'), recursive=True)
if inner_zip:
    inner_out = inner_zip[0].replace('.zip', '')
    if not os.path.exists(inner_out):
        print('reference_set.zip 해제 중...')
        with zipfile.ZipFile(inner_zip[0], 'r') as z:
            z.extractall(inner_out)
        print('완료!')
    else:
        print('이미 해제됨')

In [ ]:
# MP4 → 프레임 추출 (20장/영상 → 참조 데이터 ~4배 증가)
import cv2

FRAME_DIR        = '/content/real_frames'
FRAMES_PER_VIDEO = 20  # 5 → 20으로 증가
os.makedirs(FRAME_DIR, exist_ok=True)

video_files = glob.glob(os.path.join(REAL_DIR, '**', '*.mp4'), recursive=True)
print(f'영상 파일: {len(video_files)}개')

for vpath in tqdm(video_files, desc='프레임 추출'):
    vname = os.path.splitext(os.path.basename(vpath))[0]
    cap   = cv2.VideoCapture(vpath)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        continue
    step  = max(1, total // FRAMES_PER_VIDEO)
    saved = 0
    for fi in range(0, total, step):
        if saved >= FRAMES_PER_VIDEO:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if ret:
            cv2.imwrite(os.path.join(FRAME_DIR, f'{vname}_f{fi}.jpg'), frame)
            saved += 1
    cap.release()

real_imgs = glob.glob(os.path.join(FRAME_DIR, '*.jpg'))
print(f'실제 이미지 프레임: {len(real_imgs)}개')

In [ ]:
def find_files(base, *exts):
    files = []
    for ext in exts:
        files += glob.glob(os.path.join(base, '**', f'*.{ext}'), recursive=True)
    return sorted(files)

test_pngs = find_files(TEST_DIR, 'png')
test_wavs = find_files(TEST_DIR, 'wav')
real_wavs = find_files(REAL_DIR, 'wav')
IMG_CSV   = glob.glob(os.path.join(TEST_DIR, '**', 'Images_Detection_submission.csv'), recursive=True)[0]
AUD_CSV   = glob.glob(os.path.join(TEST_DIR, '**', 'Audio_Detection_submission.csv'),  recursive=True)[0]
IMAGE_DIR = os.path.dirname(test_pngs[0])
AUDIO_DIR = os.path.dirname(test_wavs[0])

print(f'[테스트] 이미지: {len(test_pngs)}개 | 오디오: {len(test_wavs)}개')
print(f'[실제]   이미지: {len(real_imgs)}개 | 오디오: {len(real_wavs)}개')

## 2. 이미지 딥페이크 탐지
**4중 앙상블**: CLIP KNN + CLIP 제로샷 + dima806(TTA) + FFT 주파수 분석

In [ ]:
# ── 2-1. CLIP 로드 ──────────────────────────────
from transformers import CLIPProcessor, CLIPModel

print('CLIP ViT-L/14 로드 중...')
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
clip_model.eval()
print('완료!')

In [ ]:
# ── 2-2. 제로샷 텍스트 임베딩 ───────────────────
REAL_PROMPTS = [
    'a real photograph',
    'a genuine photo taken by a camera',
    'a natural photograph of a real person',
]
FAKE_PROMPTS = [
    'an AI generated image',
    'a deepfake image',
    'a synthetic artificially generated image',
    'a computer generated fake image',
]

all_prompts = REAL_PROMPTS + FAKE_PROMPTS
text_inputs = clip_proc(text=all_prompts, return_tensors='pt', padding=True)
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

with torch.no_grad():
    text_feats = clip_model.get_text_features(**text_inputs)
    text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

real_text = text_feats[:len(REAL_PROMPTS)].mean(dim=0)
fake_text = text_feats[len(REAL_PROMPTS):].mean(dim=0)
real_text = (real_text / real_text.norm()).cpu()
fake_text = (fake_text / fake_text.norm()).cpu()
print('텍스트 임베딩 완료!')

In [ ]:
# ── 2-3. CLIP 이미지 특징 추출 함수 ─────────────
def extract_clip_features(path_list, batch_size=32, desc='CLIP'):
    all_feats, valid_idxs = [], []
    with torch.no_grad():
        for i in tqdm(range(0, len(path_list), batch_size), desc=desc):
            batch, idx = [], []
            for j, fpath in enumerate(path_list[i:i+batch_size]):
                try:
                    batch.append(Image.open(fpath).convert('RGB'))
                    idx.append(i + j)
                except:
                    pass
            if not batch:
                continue
            pv     = clip_proc(images=batch, return_tensors='pt')['pixel_values'].to(device)
            out    = clip_model.vision_model(pixel_values=pv)
            feats  = clip_model.visual_projection(out.pooler_output)
            feats  = (feats / feats.norm(dim=-1, keepdim=True)).cpu().numpy()
            for k, vi in enumerate(idx):
                all_feats.append(feats[k])
                valid_idxs.append(vi)
    return np.array(all_feats), valid_idxs

print(f'실제 이미지 {len(real_imgs)}개 처리 중...')
real_img_feats, _ = extract_clip_features(real_imgs, desc='실제 이미지')
print(f'실제 이미지 특징: {real_img_feats.shape}')

In [ ]:
img_df     = pd.read_csv(IMG_CSV)
filenames  = img_df['full_secret_name'].tolist()
test_paths = [os.path.join(IMAGE_DIR, f) for f in filenames]

print(f'테스트 이미지 {len(test_paths)}개 처리 중...')
test_img_feats, img_valid_idxs = extract_clip_features(test_paths, desc='테스트 이미지')
print(f'테스트 이미지 특징: {test_img_feats.shape}')

In [ ]:
# ── 2-4. KNN 점수 + 제로샷 점수 ─────────────────
from sklearn.neighbors import NearestNeighbors

knn_img = NearestNeighbors(n_neighbors=5, metric='cosine', n_jobs=-1)
knn_img.fit(real_img_feats)

real_self, _ = knn_img.kneighbors(real_img_feats)
img_thr      = real_self.mean(axis=1).mean() + 2.0 * real_self.mean(axis=1).std()
print(f'KNN 임계값: {img_thr:.4f}')

test_dists, _ = knn_img.kneighbors(test_img_feats)
knn_scores    = np.zeros(len(filenames))
for i, vi in enumerate(img_valid_idxs):
    knn_scores[vi] = min(test_dists[i].mean() / (img_thr + 1e-8) / 2, 1.0)

# 제로샷: 이미 추출한 특징 재사용
img_t      = torch.tensor(test_img_feats)
sim_real   = (img_t @ real_text).numpy()
sim_fake   = (img_t @ fake_text).numpy()
logits     = np.stack([sim_real, sim_fake], axis=1) * 100.0
exp_l      = np.exp(logits - logits.max(axis=1, keepdims=True))
zs_probs   = exp_l / exp_l.sum(axis=1, keepdims=True)
zs_scores  = np.zeros(len(filenames))
for i, vi in enumerate(img_valid_idxs):
    zs_scores[vi] = zs_probs[i, 1]

print(f'KNN 점수:    {knn_scores.min():.3f} ~ {knn_scores.max():.3f}')
print(f'제로샷 점수: {zs_scores.min():.3f} ~ {zs_scores.max():.3f}')

del clip_model
torch.cuda.empty_cache()
print('CLIP 해제 완료')

In [ ]:
# ── 2-5. FFT 주파수 분석 (CPU, ~5분) ────────────
def fft_fake_score(img_path):
    try:
        img = Image.open(img_path).convert('L').resize((256, 256))
        arr = np.array(img, dtype=np.float32)
        mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(arr))))
        h, w   = mag.shape
        cy, cx = h // 2, w // 2
        y, x   = np.ogrid[:h, :w]
        dist   = np.sqrt((y - cy)**2 + (x - cx)**2)
        r_low  = min(h, w) // 8
        r_high = min(h, w) // 4
        low_e  = mag[dist <= r_low].mean()
        high_e = mag[dist >  r_high].mean()
        # 고주파 비율 높을수록 Fake 가능성
        return float(high_e / (low_e + 1e-8))
    except:
        return 0.5

fft_raw = np.array([fft_fake_score(os.path.join(IMAGE_DIR, f))
                    for f in tqdm(filenames, desc='FFT 분석')])
fft_min, fft_max = fft_raw.min(), fft_raw.max()
fft_scores = (fft_raw - fft_min) / (fft_max - fft_min + 1e-8)
print(f'FFT 점수: {fft_scores.min():.3f} ~ {fft_scores.max():.3f}')

In [ ]:
# ── 2-6. dima806 추론 + TTA (좌우반전 평균) ──────
from transformers import pipeline

print('dima806 모델 로드 중...')
img_pipe = pipeline(
    'image-classification',
    model='dima806/deepfake_vs_real_image_detection',
    device=0 if torch.cuda.is_available() else -1
)
print('완료!')

In [ ]:
def get_fake_score(r):
    top   = r[0] if isinstance(r, list) else r
    label = top['label'].lower()
    score = top['score']
    return score if 'fake' in label else (1.0 - score)

model_scores = np.zeros(len(filenames))
BATCH_SIZE   = 32

for i in tqdm(range(0, len(filenames), BATCH_SIZE), desc='dima806 TTA'):
    orig_imgs, flip_imgs, batch_idx = [], [], []
    for j, fname in enumerate(filenames[i:i+BATCH_SIZE]):
        try:
            img = Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB')
            orig_imgs.append(img)
            flip_imgs.append(img.transpose(Image.FLIP_LEFT_RIGHT))
            batch_idx.append(i + j)
        except:
            pass
    if not orig_imgs:
        continue
    orig_res = img_pipe(orig_imgs)
    flip_res = img_pipe(flip_imgs)
    for k, vi in enumerate(batch_idx):
        model_scores[vi] = (get_fake_score(orig_res[k]) + get_fake_score(flip_res[k])) / 2.0

print(f'dima806 점수: {model_scores.min():.3f} ~ {model_scores.max():.3f}')
del img_pipe
torch.cuda.empty_cache()

In [ ]:
# ── 2-7. 4중 앙상블 → CSV 저장 ──────────────────
# CLIP KNN 25% + 제로샷 25% + dima806+TTA 30% + FFT 20%
final_img = 0.25*knn_scores + 0.25*zs_scores + 0.30*model_scores + 0.20*fft_scores
img_df['prediction'] = (final_img >= 0.5).astype(int)

print('이미지 예측 완료!')
print(img_df['prediction'].value_counts())

img_out = '/content/Images_Detection_submission.csv'
img_df.to_csv(img_out, index=False)
shutil.copy(img_out, '/content/drive/MyDrive/Images_Detection_submission.csv')
print('이미지 CSV Drive 저장 완료!')

## 3. 오디오 딥페이크 탐지
**2중 앙상블**: WavLM-Large KNN + wav2vec2 KNN

In [ ]:
# ── 3-1. 오디오 공통 함수 ────────────────────────
import torchaudio
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoFeatureExtractor

TARGET_SR   = 16000
MAX_SAMPLES = 16000 * 5
AUD_BATCH   = 4

def load_wav(fpath):
    wav, sr = torchaudio.load(fpath)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    return wav.mean(dim=0)[:MAX_SAMPLES].numpy()

def extract_audio_features(path_list, feat_extractor, model, desc):
    all_feats, valid_idxs = [], []
    with torch.no_grad():
        for i in tqdm(range(0, len(path_list), AUD_BATCH), desc=desc):
            batch, idx = [], []
            for j, fpath in enumerate(path_list[i:i+AUD_BATCH]):
                try:
                    batch.append(load_wav(fpath))
                    idx.append(i + j)
                except:
                    pass
            if not batch:
                continue
            inputs = feat_extractor(
                batch, sampling_rate=TARGET_SR,
                return_tensors='pt', padding=True
            ).input_values.to(device)
            feats = model(inputs).last_hidden_state.mean(dim=1).cpu().numpy()
            for k, vi in enumerate(idx):
                all_feats.append(feats[k])
                valid_idxs.append(vi)
    return np.array(all_feats), valid_idxs

def knn_classify(real_feats, test_feats, valid_idxs, n_total):
    scaler      = StandardScaler()
    real_sc     = scaler.fit_transform(real_feats)
    test_sc     = scaler.transform(test_feats)
    knn         = NearestNeighbors(n_neighbors=5, metric='cosine', n_jobs=-1)
    knn.fit(real_sc)
    self_d, _   = knn.kneighbors(real_sc)
    mu          = self_d.mean(axis=1).mean()
    sig         = self_d.mean(axis=1).std()
    thr         = mu + 2.0 * sig
    test_d, _   = knn.kneighbors(test_sc)
    scores      = np.zeros(n_total)
    for i, vi in enumerate(valid_idxs):
        scores[vi] = min(test_d[i].mean() / (thr + 1e-8) / 2, 1.0)
    print(f'  임계값: {thr:.4f} | 점수: {scores.min():.3f}~{scores.max():.3f}')
    return scores

aud_df     = pd.read_csv(AUD_CSV)
aud_names  = aud_df['full_secret_name'].tolist()
test_paths = [os.path.join(AUDIO_DIR, f) for f in aud_names]
print(f'테스트 오디오: {len(test_paths)}개 | 실제 오디오: {len(real_wavs)}개')

In [ ]:
# ── 3-2. WavLM-Large ────────────────────────────
print('WavLM-Large 로드 중...')
wavlm_fe  = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-large')
wavlm     = AutoModel.from_pretrained('microsoft/wavlm-large').to(device)
wavlm.eval()
print('완료!')

real_wl, _             = extract_audio_features(real_wavs,  wavlm_fe, wavlm, '실제 오디오 (WavLM)')
test_wl, wl_valid      = extract_audio_features(test_paths, wavlm_fe, wavlm, '테스트 오디오 (WavLM)')
print(f'WavLM 특징: 실제 {real_wl.shape} | 테스트 {test_wl.shape}')

print('WavLM KNN 분류 중...')
wavlm_scores = knn_classify(real_wl, test_wl, wl_valid, len(aud_names))

del wavlm
torch.cuda.empty_cache()
print('WavLM 해제 완료')

In [ ]:
# ── 3-3. wav2vec2-base ───────────────────────────
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

print('wav2vec2-base 로드 중...')
w2v_fe = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base')
w2v    = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base').to(device)
w2v.eval()
print('완료!')

real_w2v, _            = extract_audio_features(real_wavs,  w2v_fe, w2v, '실제 오디오 (w2v)')
test_w2v, w2v_valid    = extract_audio_features(test_paths, w2v_fe, w2v, '테스트 오디오 (w2v)')
print(f'wav2vec2 특징: 실제 {real_w2v.shape} | 테스트 {test_w2v.shape}')

print('wav2vec2 KNN 분류 중...')
w2v_scores = knn_classify(real_w2v, test_w2v, w2v_valid, len(aud_names))

del w2v
torch.cuda.empty_cache()
print('wav2vec2 해제 완료')

In [ ]:
# ── 3-4. 2중 앙상블 → CSV 저장 ───────────────────
# WavLM 60% + wav2vec2 40% (WavLM이 더 최신/강력)
final_aud = 0.60 * wavlm_scores + 0.40 * w2v_scores
aud_df['prediction'] = (final_aud >= 0.5).astype(int)

print('오디오 예측 완료!')
print(aud_df['prediction'].value_counts())

aud_out = '/content/Audio_Detection_submission.csv'
aud_df.to_csv(aud_out, index=False)
shutil.copy(aud_out, '/content/drive/MyDrive/Audio_Detection_submission.csv')
print('오디오 CSV Drive 저장 완료!')

## 4. 제출 파일 생성

In [ ]:
submit_zip = '/content/submission.zip'
with zipfile.ZipFile(submit_zip, 'w') as z:
    z.write('/content/Images_Detection_submission.csv', 'Images_Detection_submission.csv')
    z.write('/content/Audio_Detection_submission.csv',  'Audio_Detection_submission.csv')
shutil.copy(submit_zip, '/content/drive/MyDrive/submission.zip')

img_check = pd.read_csv('/content/Images_Detection_submission.csv')
aud_check = pd.read_csv('/content/Audio_Detection_submission.csv')
print('===== 최종 확인 =====')
print(f'[이미지] {len(img_check)}개 | 빈값: {img_check["prediction"].isna().sum()}')
print(img_check['prediction'].value_counts())
print(f'\n[오디오] {len(aud_check)}개 | 빈값: {aud_check["prediction"].isna().sum()}')
print(aud_check['prediction'].value_counts())
print('\n제출 파일 준비 완료!')

In [ ]:
from google.colab import files
files.download('/content/submission.zip')

In [ ]:
# Colab 임시 폴더 정리
for path in ['/content/test_data', '/content/real_data',
             '/content/real_frames', '/content/sample_data']:
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f'삭제: {path}')
total, used, free = shutil.disk_usage('/')
print(f'\n디스크 여유: {free // (1024**3)} GB')